# Master Thesis Martin Vingerhoets 

## Section 1: Introduction 

### Staggered Grid Spatial Discretization of 1D and 2D SWE  

Q1: What criteria are available to set the mesh width (i.e. the spatial resolution) in a non-linear transient wave propagation problem? Should a minimum number of grid points per wavelength or otherwise be used? How is the largest frequency obtained? How is the mesh width influence by the order of spatial discretization? Note that:

1. higher harmonics are generated (by the non-linear terms in the equation) and;
2. one wishes to preserve a minimal number of grid point per wave-length?
3. how is this question resolved in [TrixiShallowWater](https://github.com/trixi-framework/TrixiShallowWater.jl)? 

Q2: what are reference implementations of staggered grid methods 
1. see [staggered grid](https://ptsolvers.github.io/JustRelax.jl/dev/man/equations_discretization) in JustRelax.jl;

Q3: Currently the harmonics are split into sin and cos terms. Is switching to complex-valued arithmetic a feasible way to reduce overall computational cost? 

### Explicit, Implicit and Hybrid Transient Solver 

Q1: How should a [steady state solver](https://docs.sciml.ai/DiffEqDocs/stable/types/steady_state_types/) be applied to solve a transient problem? Test this first on the harmonic oscillator. 

Q2: What kind of IMEX solvers ([Split ODE Problems](https://docs.sciml.ai/DiffEqDocs/stable/types/split_ode_types/)) are available to simulate shallow water flow? What are the implicit and explicit solvers that are being combined? 

Q3: How should IMEX solvers be deployed such that all non-linear terms are treated explicitly and that merely a **linear** system needs to be solver at each time? 

Q4: How are the linear systems resulting from IMEX methods solved? Can the resulting linear system be solved using a Schur (or approximation thereof) complement method as e.g. in [shallowWaterFoam.C](https://www.openfoam.com/documentation/guides/latest/api/shallowWaterFoam_8C.html#details). 

Q5: does a linear model allow to estimate the time required to reach a steady state? 

### (Non-)Linear Time-Harmonic Solver 

Decompose time-dependent part into harmonics (base harmonic and higher order harmonic generated by the non-linear term in the equation).  

Q1: what non-linear terms are taken into account? How is the term $\| \mathbf{u} \| \mathbf{u}$ treated? How can this term be simplified without loosing too much of the physics involved?  

Q2: how can the sparse Jacobian be generated using [ForwardDiff](https://github.com/JuliaDiff/ForwardDiff.jl). How to use [sparsity dedection](https://docs.sciml.ai/NonlinearSolve/stable/tutorials/large_systems/#Approximate-Sparsity-Detection-and-Sparse-Jacobians)? See also [Sparsity handling in DifferentiationInterface](https://juliadiff.org/DifferentiationInterface.jl/DifferentiationInterface/stable/tutorials/advanced/#Sparsity). 

Q3: what is the structure of the Jacobian? How to split Jacobian into a linear part (independent of the Newton step) and an non-linear part (varrying with the  Newton step). A3: see draft text Martin (Domenico needs to read);  

Q4: how to store the Jacobian as a [block matrix](https://juliaarrays.github.io/BlockArrays.jl/stable/man/blockarrays/) rendering the block preconditioner easier to implement? See [conversation with Guillaume Dalle on slack](https://julialang.slack.com/archives/C6A044SQH/p1774212987002119). 

Q5: how to convert coupled first order system as a second order system, thus making it feasible to replace the ILU preconditioner by the AMG preconditioiner; 

Q6: how to apply a conformal transformation on the linear part of the Jacobian to obtain all eigenvalues on the same side of the imaginary axis? 

Q7: how to freeze the preconditioner over various iterations on the Newton iteration? 

### Newton-LU for both Transient and Time-Harmonic Solvers

<b>Advantages of LU:</b> Robust. Mainstream, and thus well documented. What about modern variatiants (e.g. Pardiso), recursive factorizations or GPU implementations?    

<b>Disadvantages of LU:</b>

1. the large number of fill-in elements, especially in case of two-dimensional discretization with large number of harmonics;
2. solving too accuratey (oversolving) especially at first Newton iterations, when still far from the final solution (cfr. forcing terms, sufficient descent directions);

<b>Pointers to Direct Solver that Exploit Sparsity</b>

1. the forward and backward substitution is implemented in <i>ldiv!</i>;
2. the sparse symmetric and positive definite case is implemented in <i>CHOLMOD</i>. See [this Discourse post](https://discourse.julialang.org/t/defining-a-preconditioner-for-iterativesolvers/86977/17);  

### Newton-Krylov for both Transient and Time-Harmonic Solvers

<b>Advantages of Krylov:</b> Avoid oversolving using [forcing_strategies](https://docs.sciml.ai/NonlinearSolve/stable/native/solvers/#forcing_strategies). 

<b>Disadvantages of Krylov:</b> Slow to converge in case that the eigenvalues of the Jacobian spread accross both sides of the imaginary axis unless properly treated (using a conformal mapping as in the case of the complex-shifted Laplace preconditioner). 

Q1: How does one ensure that the GMRES search space is allocated once only, accross all of the Newton iterations? LinearSolve.jl builds it once in its cache init and then future solves just reinit!. 

Q2: how does the GMRES solver converge? What is the influence of the restart value on the overall computational cost? Is switching to a short-term Krylov solver (BiCGSTAB, IDR(s) or friend borrowed from petsc.jl) a feasible way to reduce overall computational cost?  

Q3: how to specify the preconditioner in the example on the page [forcing strategies](https://docs.sciml.ai/NonlinearSolve/stable/native/solvers/#forcing_strategies).

Q4: what does <i>reuse_A_if_factorization</i> refer to? 

References: book by Tim Kelley on Non-Linear Equations, papers by e.g. Dembo-Eisenstat; 

### Packages for Krylov Subspace Solvers 

1. [KrylovKit.jl](https://jutho.github.io/KrylovKit.jl/stable/man/linear/)
2. [Krylov.jl](https://jso.dev/Krylov.jl/stable/) In particular [restarted GMRES](https://jso.dev/Krylov.jl/stable/solvers/unsymmetric/#GMRES)
3. [iterativesolvers](https://iterativesolvers.julialinearalgebra.org/dev/)

### GMRES Solvers 

1. restart value (and loss of superlinear converge due to restart)?
2. initial guess (take e.g. solution from previous Newton iteration, from previous water depth value or from coarser mesh)?
3. stopping criterium (how many GMRES iterations can be solved by implementing forcing terms?)?
4. compare GMRES with alternative Krylov subspace solver that exploit short term recurrences?   

### Scalar ILU($\tau$) Preconditioners 

How does ILU out of the box perform? Can the ILU decompositon of the Jacobian froozen over various Newton iterations.   

### AMG Solver 

Q1: how does the AMG solver converge? Currently AMG is used as a preconditioner for GMRES. Does AMG used as a solver instead, i.e., without the use of an outer Krylov iteration (i.e., without the use of an outer GMRES iteration) still converge. What is the asymptotic rate of convergence of this solver? How does this rate depend on factors such as the grid density (number of degrees of freedom), the amount of linear wave damping, the number of grid points per wavelength, the number of pre and post smoothing steps, and the type of cycle (V-cycle vs. W-cycle). 

Q2: suppose switching the smoother inside AMG off, i.e., setting pre and post smoothing steps both to zero (this amounts to using the coarse grid correction as a preconditioner, i.e., using multigrid deflation). Does the outer GMRES iteration still converge? If not, by how much is convergence delayed? Is switching the smoother off a feasible way to reduce overall computational cost? 

Q3: what is the computational cost of the AMG solver? How do the computational cost of the AMG set-up and solve phase compare? What factors influence the computational cost of the set-up phase? Suppose that the speed of coarsening is defined as the reduction of degrees of freedom from a level to the next coarser level? Can the  speed of coarsening accuracy be influenced? Can a fast and therefore inaccurate coarsening be compensated by an outer GMRES (or other Krylov) iteration? Is switching to a fast and inaccurate coarsening a feasible way to reduce overall computational cost? (cfr. literature on aggressive coarsening and multi-pass interpolation)

Q4: the implementation AlgebraicMultigrid.jl provides both a Ruge-Stueben and smoothed aggregation variant to algebraic multigrid. Which one did you use? How the variant you used compare with its counterpart. Does the smoothed aggregation approach allow to accelerate the coarsening and reduce the computational cost of the application of the preconditioner. 

Q5: Implementation of AMG. Is there a benefit to switching the AMG solver that the PETSc library provides. See petsc.jl.  

### Block Jacobi Preconditioners    

Assume a spatial discretization using $N_c$ cells (or elements or any alternative denoting the spatial accuracy) and using $N_h$ harmonics. Then the Jacobian is of size $N_c*N_h$. Martin showed the Jacobian as $N_c$ blocks, of size $N_h$. We instead would like to see the Jacobian as $N_h$ blocks of size $N_d$ (thus arriving at a smaller number of blocks that are large in size, especially in case of two-dimensional discretizations at high spatial resolution). 

1. preconditioners [BlockArrays](https://github.com/JuliaArrays/BlockArrays.jl)
2. [BlockFactorizations](https://github.com/SebastianAment/BlockFactorizations.jl) provides allocation-free matrix vector products;

### Parallel in Frequency using pmap 

Can the $N_h$ blocks of size $N_c$ be solved independently from each other using the function <i>pmap</i>? 

### Spatial Coarse Grid to Curb Increase in GMRES Iterations as $N_h$ Increases  

Can the increase in GMRES iterationss with $N_h$ bew curbed using coarser grids (lower spatial resolution).

## Section 2: Bifurcation Analysis 

Solve for gradually smaller height of the channel (as the non-linearity is proportional to inverse of height).  

## Section 3: Iterative Solvers - A Closer Look 

### Section 1.3: Poisson Equation

See notebook [amg_runs](./amg_runs.ipynb). 

Solve (note the minus sign introduced to obtain matrix with a positive diagonal) $$ - \bigtriangleup u = f $$. 

<b>AlgebraicMultigrid.jl as solver</b>

* what is the asymptotic rate of convergence of the solver?
* how does the asymptotic rate of convergence depend on the solver settings, such as the number of pre and post smoothing steps, the cycle type (V-cycle vs. W-cycle);
* what is the advantage of using a high number (4 or higher) of smoothing steps;
* repeat above for $- \bigtriangleup u + \beta u = f$ assuming $\beta>0$ (positive real shift, increasing weight on the diagonal)

<b>AlgebraicMultigrid.jl as preconditioner for the Conjugate Gradient Method</b>

* what is the advantage of using AlgebraicMultigrid.jl as a preconditioner for the conjugate gradient method?
* can the number of smoothing steps be reduced? Entirely eliminated, thus using the coarse grid correction only as preconditioner?

<b>Implementation aspects of AlgebraicMultigrid.jl</b>

* what is the number of allocations in the AlgebraicMultigrid.jl setup and solve phase; 

<b>PCGAMG in PETSc.jl as alternative for AlgebraicMultigrid.jl</b>

* use [PCGAMG](https://petsc.org/release/manualpages/PC/PCGAMG/) as alternative AMG implementation. Any computational advantageous? 

### Section 2.3: Poisson Equation with a Purely Imaginary Shift 

Solve $$ - \bigtriangleup u + \iota \beta u = f $$. 

* what happens in case of large shift $\beta$?
* comlex 

References: 
* D. Lahaye, H. De Gersem, S. Vandewalle and K. Hameyer, {\em Algebraic multigrid for complex symmetric systems}, IEEE Trans. on Magn., Volume 36, Issue 4, July 2000; 
* Roland Freund, {\em Conjugate Gradient-Type Methods for Linear Systems with Complex Symmetric Coefficient Matrices}, SIAM Journal on Scientific and Statistical Computing 13.1, 425--448, 1992.  

### Section 3.3: Helmholtz Equation 

Solve $$ - \bigtriangleup u + (1 + \iota \beta) k^2 u = f $$. 

* what is the influence of number of grid points per wavelength?
  
### Section 4.3: Scalar Wave Equation with Cubic Damping 

See notebook [nonlinear-wave-equation](./nonlinear-wave-equation.ipynb)

Solve 

$$
\frac{\partial^2 \, u}{\partial t^2} 
+ \gamma \frac{\partial \, u}{\partial t} 
+ \gamma_3 \left( \frac{\partial \, u}{\partial t} \right)^3 
= c^2 \frac{\partial^2 \, u}{\partial x^2} + f(x,t) 
$$ 

using the harmonic balance method. 

* what is the influence of $\gamma_3$? 

## Section 4: Draft Paper 

* Tentative title: A Newton - Krylov - Algebraic Multigrid Solver for Tidal Shallow Water Flow in Frequency Domain 
* Authors: MV, HS and DL 
* Scientific contribution: block preconditioner for Jacobian linear system. Blocks group degrees of freedom per harmonic component. Blocks can be approximately inverted using algebraic multigrid after introducing a sufficient amount of damping. Matrix-free Krylov implementation using  forward mode automatic differentiation.  Multi-threaded implementation using parallel processing in Julia programming language. 

### Section 1: Introduction 

<b>Problem description</b>: We consider the numerical simulation of two-dimensionsal shallow water flow (depth-averaged height and velocity) with periodic (tidal) forcing. (Elaborate on the local equilibrium approach. Cite appropriate sources.) The conservation of momentum equation contains non-linear terms that model the bed friction. These non-linear terms cause higher harmonic to arrise. 
 
<b>State of the Art</b>: Two classes of solution methods do exist: a multi-frequency and time-domain approach. The multi-frequency results in large sparse linear systems that are challenging to solve. Therefore a transient (time-stepping) approach is typically preferred. The transient approach, however, fails to closer to the physical properties of the periodic solutions that are vital to obtain in practise.  

<b>Harmonic Balance Method for Tidal Shallow Water Flow</b>: Pose multi-harmmonic anzatz for height and velocity components. Plug into hydrodynamics model, work out terms, balance the harmonics and discretize in space. Resulting non-linear systems of equations. If solved using a Newton, the Jacobian linear is challenging to solve due to the all-to-all coupling of the amplitudes. 

<b>Scientific Contribution</b>: To address this problem, we developed a computationally efficient Newton-Krylov solver for the multi-frequency formulation. To develop the preconditioner, we rely on concepts from domain decomposition methods and Helmholtz equation solvers. We group the components of the solution vector that correspond to a particular frequency into blocks. These blocks correspond to subdomains in the domain decomposition method. The block reordering of the vector induces a block-reordering of the Jacobian linear system at each Newton iteration step. We precondition this Jacobian by approximately inverting the diagonal Helmholtz operator blocks in an additive sequence. This corresponds to solving subdomains problems in an additive Schwartz method. These subdomain problems can efficiently be inverted using an algebraic multigrid method after introducing a suffcient amount of damping.

The use of the Julia programming language allows us to compute the Jacobian by automatic differentiation, to implement a matrix-free Krylov acceleration, to Taylor a previously existing algebraic multigrid solver to our needs, and to implement the local solves in a thread-parallel method.  

<b>Numerical Results Obtained</b>: Quantify computational efficiency obtained.  

<b>Outside Scope</b>
* 2D rectangular domains only;
* no morphodynamics; 

<b>Structure of Paper</b>: This paper is structured as follows. 

### Section 2: Tidal Shallow Water Flow

In this section, we introduce the two-dimensional depth-averaged shallow water flow with tidal (periodic) forcing and non-linear bottom friction terms. 

The hydrodynamics model expresses the convervation of mass and the conservation of momentum in terms of the height and two velocity components. Rectangular computational domain consider to be sufficient to demonstrate the essential feastures, without loss of general setting. Specify the driving frequency $\omega_d$ and the spatial variation of the tidal forcing. Specify that the strength of the non-linear friction varries with the deptrh of the channel. Specify periodic boundary conditions and initial conditions. 

<div class="alert alert-block alert-success"> PDE + BC + IC from MV's draft. </div>

Commonly used numerical approximation: staggered finite difference discretization in space followed by a time-integration method (explicit, implicit or hybrid). This approach, however, fails to take the periodic forcing of the flow into account. Physical information that may guide the numerical solution is lost.  

Representative numerical solutions for representative test cases can be found in (cite references here). 

### Section 3: Non-Linear Harmonic Balance Method  

In this section, we first describe the harmonic balance method applied to the linear shallow water equations in a rectangular channel with periodic forcing. We subsequently extend this discussion to the non-linear case. (In this section, we assume that the harmonic balancing is applied prior to the spatial discretization. Check with MV that this is indeed what he does).

<b>Linear Case</b> In the linear case (in which the non-linear friction can be neglected, deep channels), the harmonic balance method assumes the solution to inherit its temporal dependency from the forcing. This allow to assume that $u(x,t) = \widehat{u}(x) \, \exp(j \omega_d t)$ (for height and velocity components). This in turn reduces the space-time problem for $u(x,t)$ to a space-only problem for $\widehat{u}(x)$ (replacing ddt by j-times-omega). The linear shallow water equations can be reformulated as a Helmholtz eqiuation for the water height. After spatial discretization, the linear algebra problem can be efficiently solved a Krylov method preconditioned by introducibng some form of damping. 
   
<b>Non-Linear Case</b> Next, we extend the above discussion to the non-linear case in which non-linear friction csan no longer be neglected (channel small depth).

1. expand time-dependency of numerical approximation of the height and the velocity fields as a truncated Fourier series with a limited number of harmonics (instead of merely one harmonic as in the linear case). This number is a parameter in the numerical approximation (equivalent to a time-step). The amplitudes become the functions to be approximated;
2. substitute the truncated Fourier series in the shallow water equations. Expand the non-linear term, compute the amplitudes of the higher order harmonics, perform harmonic balancing and arrive a system of coupled differential equations for the harmonic amplitudes; 
3. discretize the resulting system of partial differentil equations in space using Arakawi-C staggered grids for pressure-velocity coupling for the various modes. 

The results in a system of non-linear equation for the numerical approximation to the harmonic amplitudes. This is a system of coupled Helmholtz equation. Solved by a globalized Newton method with an exact Jacobian. Resulting linear sytem is large and has a non-standard structure due to the cross-coupling of the various harmonics.    

<div class="alert alert-block alert-success"> Jacobian structure from MV's draft.</div>

### Section 4: Newton - Krylov - Algebraic-Multigrid Method  

In this section, we provide details of how the Jacobian linear systems is solved by a preconditioner Krylov method at each Newton iteration. 

Matrix-free formulation using forward-mode automatic (code) differentation. Block preconditioning. Algebraic multigrid coarsening and Galerkin coarse grid operator ahead of the Newton iteration. Damped Helmholtz system. Algebraic multigrid approximate solve. 

<div class="alert alert-block alert-success"> Newton-GMRES convergence per frequency to demonstrate that residual is properly distributed among components from MV's draft. </div>

### Section 5: Numerical Results  

In this section, we provide numerical results that demonstrate the computational efficiency of the solver we propose. (Describe the various subsections). 

<div class="alert alert-block alert-success"> Here a figure showing the harmonic content of the signal (obtained after FFT) for various amplitudes of the non-linear damping allowing to illustrate that a larger non-linear damping results in larger high frequency harmonics. </div>

<div class="alert alert-block alert-success"> Here a figure showing the convergence of the non-linear residual during the Newton - GMRES iteration for the various harmonics allowing to emphasize that residual is properly distributed over the various harmonics. </div>

<div class="alert alert-block alert-success"> Here a figure or table with representative CPU timeas (details to be established). </div>

### Section 6: Conclusions 

### References 

* Suitable references for tidal shallow water flow, its local equilibrium formulation (expression for the forcing $F_d$), its spatial discretization and its solution in time-domain;

* Suitable references for the harmonic balance methods applied to ODEs and PDEs;  

* Suitable references for globalized Newton method: book Tim Kelley, book Toine; 
  
* Suitable references for Krylov method and ILU preconditioners: book Saad; 

* Suitable references for multigrid solver for damped Helmholtz equation: paper Lahaye-deGersem, paper Erlangga;

* Suitable reference for algebraic multigrid: Chapter by Klaus Stuebern in book by Scheuller-Trottenberg-Oosterlee; 
  
* Suitable references for the Julia packages used. These include NonlinearSolvers.jl, LinearAlgebra, SparseArrays, FFTW, Statistics, Plots, Measures, ADTypes, Dates, Profile, Serialization, AlgebraicMultigrid, LinearSolve, ParU_jll, PolyesterForwardDiff and others. 

## Section 4: Dialoog met Martin 



## References 

Hierbij de link naar de files met het model:

https://filesender.surf.nl/?s=download&token=a7bd1605-4e56-485e-96e3-6d1750ba2572

Ik heb nog even gevraagd hoe de auteur (Haoyan) de niet-lineaire term doet. Hij doet inderdaad wat Martin voorstelde:

Mijn vraag: Do you substitute the velocity signal from the previous iteration and then do a FT, or do you have a 'closed' form solution?

Antwoord:

I followed the first approach: substituting the velocity signal from the previous iteration and then applying the Fourier transform.

As shown in Equation S.14, \phi^{j-1} indicates that it comes from the previous iteration.

Misschien kan dit slimmer (Domenico weet misschien een truc), maar deze werkt iig. Weet niet of het slim is om te discretiseren, ik denk dat het beter is een functie te maken (met de coefficienten uit de vorige iteratie), die te vermenigvuldigen met een specifieke mode (gebruik orthogonaliteit) en die dan met een slimme integrator te evalueren. Kunnen we het volgende week over hebben.

Merk trouwens ook op dat Haoyan niet vermenigvuldigd met (H+\zeta-h) maar die samen met de |u| numeriek evalueert zoals boven geschreven.